# READ ME

* The source data that this notebook is meant to parse and dump into a csv for training originates from the repo which as of 4/18/26 resides in `data/spectra/raw/nist_IR.zip`
  * https://github.com/IvanChernyshov/NistChemData


### Purpose
* the purpose of this file is to parse the jdx files into a structured data format like a csv
  * uses the jcamp library to parse the file 

### Output
* jdx_metadata.csv contains data on the compound id, compound name, a few compound properties, data on the spectroscopy equipment and techniques used, and a few descriptive statistics of the spectroscopy data
* jdx_xy_points.csv contains the spectroscopy data as a pair of cordinates (x,y) where x is the wavelength and y is the transmitance 
  * logic for parsing such contained in `expand_xy_points`

In [1]:
from pathlib import Path
from typing import Any

from IPython.display import display
import numpy as np
import pandas as pd
from jcamp import jcamp_read


## Parse JDX into csv

## funcitons to parse jdx using jcamp

In [2]:
def as_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, np.ndarray):
        return ""
    if isinstance(value, (np.integer, np.floating)):
        return str(value.item())
    return str(value).replace("\r\n", "\n").replace("\r", "\n").replace("\n", " ").strip()


def as_number(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, (np.integer, np.floating)):
        return str(value.item())
    if isinstance(value, (int, float)):
        return str(value)
    return as_text(value)


def expand_xy_points(data: dict[str, Any]) -> pd.DataFrame:
    """Return explicit x/y pairs for the spectrum.

    JCAMP files often encode an IR trace as a starting x value plus a delta x,
    followed by a list of y values. For example, a line like:

        549.759 29.3 29.05 28.32 ...

    means:
        (549.759, 29.3)
        (549.759 + deltax * 1, 29.05)
        (549.759 + deltax * 2, 28.32)

    The `jcamp` reader usually gives us explicit `x` and `y` arrays already.
    If those arrays are missing, we rebuild x from `firstx` and `deltax`.
    """

    x = data.get("x")
    y = data.get("y")

    if x is not None and y is not None and len(x) == len(y) and len(x) > 0:
        return pd.DataFrame({"x": np.asarray(x, dtype=float), "y": np.asarray(y, dtype=float)})

    if y is None or len(y) == 0:
        return pd.DataFrame(columns=["x", "y"])

    y_values = np.asarray(y, dtype=float)
    firstx = data.get("firstx")
    deltax = data.get("deltax")

    if firstx is not None and deltax is not None:
        x_values = float(firstx) + float(deltax) * np.arange(len(y_values))
    else:
        x_values = np.arange(len(y_values), dtype=float)

    return pd.DataFrame({"x": x_values, "y": y_values})


def parse_jdx_file(path: Path) -> tuple[dict[str, str], pd.DataFrame]:
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        data = jcamp_read(handle)

    metadata: dict[str, str] = {"filename": path.name}
    for key, value in data.items():
        if key in {"x", "y"}:
            continue
        if isinstance(value, np.ndarray):
            continue
        metadata[key] = as_text(value)

    x = data.get("x")
    y = data.get("y")
    metadata["npoints"] = as_number(data.get("npoints", len(x) if x is not None else ""))
    metadata["x_min"] = as_number(np.min(x) if x is not None and len(x) else "")
    metadata["x_max"] = as_number(np.max(x) if x is not None and len(x) else "")
    metadata["y_min"] = as_number(np.min(y) if y is not None and len(y) else "")
    metadata["y_max"] = as_number(np.max(y) if y is not None and len(y) else "")
    metadata["firstx"] = as_number(data.get("firstx"))
    metadata["lastx"] = as_number(data.get("lastx"))
    metadata["deltax"] = as_number(data.get("deltax"))

    spectrum = expand_xy_points(data)
    if not spectrum.empty:
        spectrum.insert(0, "filename", path.name)
        spectrum.insert(1, "point_index", np.arange(len(spectrum), dtype=int))
    else:
        spectrum = pd.DataFrame(columns=["filename", "point_index", "x", "y"])

    return metadata, spectrum


### Get list of jdx files


In [7]:
jdx_dir = Path.cwd() / "data" / "IR"
if not jdx_dir.exists():
    alt_dir = Path.cwd().parent / "data" / "IR"
    if alt_dir.exists():
        jdx_dir = alt_dir

jdx_files = sorted(jdx_dir.glob("*.jdx"))
print("jdx dir:", jdx_dir)
print("jdx files are:", len(jdx_files))
jdx_files[:5]


jdx dir: /home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR
jdx files are: 19582


[PosixPath('/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR/B6000033_IR_0.jdx'),
 PosixPath('/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR/B6000034_IR_0.jdx'),
 PosixPath('/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR/B6000036_IR_0.jdx'),
 PosixPath('/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR/B6000043_IR_0.jdx'),
 PosixPath('/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR/B6000044_IR_0.jdx')]

### parse each file and concatonate into csv

In [8]:
metadata_rows: list[dict[str, str]] = []
xy_frames: list[pd.DataFrame] = []
failed_files: list[dict[str, str]] = []

for path in jdx_files:
    try:
        metadata, spectrum = parse_jdx_file(path)
        metadata_rows.append(metadata)
        xy_frames.append(spectrum)
    except Exception as exc:
        failed_files.append({"filename": path.name, "error": f"{type(exc).__name__}: {exc}"})

metadata_df = pd.DataFrame(metadata_rows)
xy_df = pd.concat(xy_frames, ignore_index=True) if xy_frames else pd.DataFrame(columns=["filename", "point_index", "x", "y"])
failed_df = pd.DataFrame(failed_files)

metadata_df.head()


,filename,title,jcamp-dx,data type,class,origin,owner,date,names,molform,...,sphere diameter,acquisition mode,coadded scans,phase resolution,zerofilling,spectral resolution,wavenumber accuracy,apodization function,low pass filter,switch gain on
0,B6000033_IR_0.jdx,"ETHANOL, 2,2'-(5-CHLORO-2-ETHOXY PHENYLIMIDO) DI",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","ANILINE, 2-ETHOXY-5-CHLORO-N,N-(2,2'-DIETHANOL)",C12 H18 Cl N O3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B6000034_IR_0.jdx,2-(O-CHLOROPHENYL)BENZOXAZONE-4,4.24,INFRARED SPECTRUM,COBLENTZ,NaN,COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","2-(2-chlorophenyl)-2,3-dihydro-4H-1,3-benzoxaz...",C14 H10 N O2 Cl,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,B6000036_IR_0.jdx,2-ISOBUTYLBENZOXAZONE-4,4.24,INFRARED SPECTRUM,COBLENTZ,NaN,COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970","2-isobutyl-2,3-dihydro-4H-1,3-benzoxazin-4-one",C12 H15 N O2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,B6000043_IR_0.jdx,"SULFONYL-O,O'-DIPHENACYLDIBENZOATE",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970",2-[3-((3-[(benzoyloxy)acetyl]phenyl)sulfonyl)p...,C30 H22 O8 S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,B6000044_IR_0.jdx,"SULFONYL-O,M'-DIBENZOIC ACID, DIPHENACYL ESTER",4.24,INFRARED SPECTRUM,COBLENTZ,"TENNESSEE EASTMAN COMPANY, RESEARCH LABORATORIES",COBLENTZ SOCIETY Collection (C) 2018 copyright...,"Not specified, most likely prior to 1970",NaN,C30 H22 O8 S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
xy_df.head()


,filename,point_index,x,y
0,B6000033_IR_0.jdx,0,2.035,0.751
1,B6000033_IR_0.jdx,1,2.041613,0.751
2,B6000033_IR_0.jdx,2,2.048225,0.751
3,B6000033_IR_0.jdx,3,2.054838,0.751
4,B6000033_IR_0.jdx,4,2.06145,0.751


### Output into csv

In [10]:
metadata_out = Path("data/jdx_metadata.csv")
xy_out = Path("data/jdx_xy_points.csv")

metadata_out.parent.mkdir(parents=True, exist_ok=True)
metadata_df.to_csv(metadata_out, index=False)
xy_df.to_csv(xy_out, index=False)

print(f"Wrote {len(metadata_df)} metadata rows to {metadata_out}")
print(f"Wrote {len(xy_df)} xy rows to {xy_out}")

if not failed_df.empty:
    display(failed_df)


Wrote 19581 metadata rows to data/jdx_metadata.csv
Wrote 59986786 xy rows to data/jdx_xy_points.csv


,filename,error
0,B6000446_IR_0.jdx,Exception: Unknown character (t) encountered w...


## Normalize compound id

In [ ]:
import cirpy

## Join spectroscopy metadata with xy points  